# Sample ecommerce overview

Dashboard exported from example-mcp-dashbuilder


In [ ]:
from elasticsearch import Elasticsearch
import pandas as pd
import matplotlib.pyplot as plt

es = Elasticsearch(
    "http://localhost:9200",
    basic_auth=("elastic", "changeme"),
)


## Total revenue (USD)

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS total_revenue = ROUND(SUM(taxful_total_price), 2)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['total_revenue']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Total revenue (USD)</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


In [ ]:
# Total revenue (USD) — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS revenue = SUM(taxful_total_price) BY day = BUCKET(order_date, 1 day)
    | SORT day
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Total revenue (USD) — Trend")
plt.tight_layout()
plt.show()


## Orders

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS order_count = COUNT(*)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['order_count']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Orders</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


In [ ]:
# Orders — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS orders = COUNT(*) BY day = BUCKET(order_date, 1 day)
    | SORT day
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Orders — Trend")
plt.tight_layout()
plt.show()


## Average order value

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS avg_order = ROUND(AVG(taxful_total_price), 2)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['avg_order']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Average order value</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


In [ ]:
# Average order value — trend sparkline
trend_result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS avg_order = ROUND(AVG(taxful_total_price), 2) BY day = BUCKET(order_date, 1 day)
    | SORT day
    """,
    format="json"
)

trend_df = pd.DataFrame(trend_result["values"], columns=[c["name"] for c in trend_result["columns"]])
trend_df.plot(title="Average order value — Trend")
plt.tight_layout()
plt.show()


## Unique customers

Chart type: **metric**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | STATS customers = COUNT_DISTINCT(customer_id)
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
from IPython.display import HTML, display
value = df.iloc[0]['customers']
display(HTML(f'''
<div style="padding: 20px 24px; border: 1px solid #e0e0e0; border-radius: 8px; background: #fafafa; display: inline-block; min-width: 220px;">
  <div style="font-size: 13px; color: #666; text-transform: uppercase; letter-spacing: 0.5px; margin-bottom: 8px;">Unique customers</div>
  <div style="font-size: 42px; font-weight: 600; color: #1a1a1a; line-height: 1;">{value}</div>
</div>
'''))


## Order volume by day of week and hour

Chart type: **heatmap**


In [ ]:
result = es.esql.query(
    query="""
    FROM kibana_sample_data_ecommerce
    | EVAL day = DATE_FORMAT("EEEE", order_date), hour = DATE_FORMAT("HH", order_date)
    | STATS order_count = COUNT(*) BY day, hour
    | SORT day, hour
    """,
    format="json"
)

df = pd.DataFrame(result["values"], columns=[c["name"] for c in result["columns"]])
pivot = df.pivot_table(index="day", columns="hour", values="order_count")
fig, ax = plt.subplots()
im = ax.imshow(pivot.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels(pivot.columns, rotation=45, ha="right")
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels(pivot.index)
for i in range(pivot.shape[0]):
    for j in range(pivot.shape[1]):
        ax.text(j, i, f"{pivot.values[i, j]:.0f}", ha="center", va="center")
fig.colorbar(im, ax=ax)
ax.set_title("Order volume by day of week and hour")
plt.tight_layout()
plt.show()
